In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mssm.src.python.gamm_solvers import compute_S_emb_pinv_det,getImplicitV
from mssm.models import *
from mssmViz.plot import *
from mssmViz.sim import *
import matplotlib
import copy as copy
from matplotlib import font_manager
import dotenv
import os
try:
    from src.tweedielsr import Tweedie
except Exception as e:
    warnings.warn("Could not load Tweedie distribution. Is rpy2 installed correctly?")
dotenv.load_dotenv()

# Optionally load fonts (plots want "Source Sans 3")
font_path = os.getenv("font_path")
n_cores = int(os.getenv("n_cores"))

if font_path is not None:
    font_files = font_manager.findSystemFonts(fontpaths=font_path)
    for font_file in font_files:
        font_manager.fontManager.addfont(font_file)

# Some settings for plots
cmp = matplotlib.colormaps['RdYlBu_r']
plt.rcParams["font.family"] = "Source Sans 3"
plt.rcParams["font.weight"] = "semibold"
plt.rcParams["font.size"] = 8
plt.rcParams["axes.titlesize"] = 9.5
plt.rcParams["axes.labelsize"] = 8
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["figure.titlesize"] = 11
math_font_size = 8
math_font = 'cm'

seed = 42*3
np_gen = np.random.default_rng(seed)

size_conv = 2.54
single_width = 6/size_conv
double_width = 12/size_conv
full_width = 19/size_conv

## Massive Additive Mixed Model

In [ ]:
# Simulate data
n_ranef = 5000
n_obs = 200
n_dat = n_ranef*n_obs

# Can take a while..
mixed_dat = sim11(n_dat,2,c=0,seed=seed,family=Gaussian(),
                  n_ranef=n_ranef)

# Fit model with N_u=30
bfgs_options = {"gtol": 0,
                "ftol": 1e-7,
                "maxcor": 30,
                "maxls": 100,
                "maxfun": 5000,
               }

mixed_formula_m = Formula(lhs("y"),
                         [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                         data=mixed_dat)

mixed_model = GSMM([mixed_formula_m],family=GAMLSSGSMMFamily(1,Gaussian()))
mixed_model.family.init_coef = lambda x: None # Default relies on QR decomposition.. too expensive!
mixed_model.fit(method='qEFS',
                bfgs_options=bfgs_options,
                structured_qefs=False,
                sample_hessian=False,
                n_cores=1,
                form_VH=False,
                sample_hessian_options={"n_samples":30},
                qEFS_final_memory_usage=None)

# Placeholder to make mssmViz plotting code work.
mixed_model.lvi = scp.sparse.eye_array(len(mixed_model.coef),format='csc')

In [ ]:
fig = plt.figure(figsize=(full_width,0.75*single_width),layout='constrained')
axs = fig.subplots(1,5,gridspec_kw=dict(wspace=0.01,hspace=0.01))
# Plot true vs estimated means
idx = np_gen.choice(mixed_dat.shape[0],size=1000)
axs[0].scatter(mixed_model.mus[0].flatten()[idx],mixed_dat["eta"].values[idx],color="black",s=1)
axs[0].spines['top'].set_visible(False)
axs[0].spines['right'].set_visible(False)
axs[0].spines['left'].set_visible(True)
axs[0].set_ylabel("$\\mu$",math_fontfamily=math_font,)
axs[0].set_xlabel("$\\hat{\\mu}$",math_fontfamily=math_font,)

# Prepare computation of CIs via implicit representation of coef. covariance matrix approximation V
colsH = mixed_model.coef.shape[0]
S_emb, _, _, _ = compute_S_emb_pinv_det(colsH, mixed_model.overall_penalties, "svd")
V0, t1, t2, t3, _, _, _, _, _, _ = getImplicitV(mixed_model.lvi_linop,
                                                colsH, S_emb, n_c=n_cores)

# Prepare sample data to evaluate function estimates
x = np.linspace(0,1,30)
pred_dat = pd.DataFrame({"x0":x,
                         "x1":x,
                         "x2":x,
                         "x3":x,
                         "x4":[mixed_dat["x4"].values[0] for _ in range(len(x))]})


alpha = 0.05
for ti,t in enumerate([1,2,3]):

    pred = mixed_model.predict([t],n_dat=pred_dat)

    Xp = pred[1]
    Xp = scp.sparse.hstack([Xp,np.zeros(Xp.shape[0]).reshape(-1,1)],format="csc")
    

    c = Xp @ ((V0 @ Xp.T) - (t1 @ t2 @ (t3 @ Xp.T)))
    c = c.diagonal()
    b = scp.stats.norm.ppf(1 - (alpha / 2)) * np.sqrt(c)
    ciu = pred[0] + b
    cil = pred[0] - b

    axs[ti+1].plot(x,pred[0],color=cmp(0.7))

    axs[ti+1].fill([*x,*np.flip(x)],
                  [*ciu,*np.flip(cil)],color=cmp(0.7),alpha=0.5)
    
    axs[ti+1].spines['top'].set_visible(False)
    axs[ti+1].spines['right'].set_visible(False)
    axs[ti+1].spines['left'].set_visible(True)
    axs[ti+1].set_ylabel("$f(" + f"x{t-1}" + ")$",math_fontfamily=math_font,)
    axs[ti+1].set_xlabel(f"x{t-1}",fontweight='bold')

# Plot random wiggly curves
plot(mixed_model,ci=False,which=[5],axs=[axs[4]],seed=seed)

for axi,label in enumerate(["a)","b)","c)","d)","e)"]):
    axs[axi].text(-0.3,1.05,label,transform=axs[axi].transAxes,
                  fontweight='bold',fontsize=plt.rcParams["axes.titlesize"])

plt.savefig("./results/plots/mixed_model.pdf",format="pdf", bbox_inches='tight')
plt.show()

## HSMM of Energy Prices

In [ ]:
# Load data
energy_dat = pd.read_csv("./data/energy.csv")

# Create pseudo dat for latent process model
d_dat = pd.DataFrame({"pseudo": np.arange(1)})

# Setup hsmm via mssm
o_family = GAUMLSS()
d_family = GAMMALS()
n_S = 2 # Number of latent states
M = 1 # Number of observed signals
D = 100 # Max duration assumed for latent states
starts_with_first = True # First state starts with first observation?
links = []
obs_families = []
obs_formulas = []
d_families = []
d_formulas = []
pars = 0

for j in range(n_S):
    obs_families.append([])

    for m in range(M):
        obs_families[-1].append(o_family)
        # Model of mean of obs model in state j and of signal m
        obs_formulas.append(Formula(lhs("Price"), [i(),f(["EurDol"])], data=energy_dat))

        # Model of scale parameter of obs model in state j and of signal m is the same
        obs_formulas.append(Formula(lhs("Price"), [i(),f(["EurDol"])], data=energy_dat))

        links.extend(o_family.links)
        pars += 2

for j in range(n_S * (1 if starts_with_first else 2)):
    d_families.append(d_family)

    # Model of mean of dur model in state j
    d_formulas.append(Formula(lhs("pseudo"), [i()], data=d_dat))

    # Model of scale parameter of dur model in state j
    d_formulas.append(Formula(lhs("pseudo"), [i()], data=d_dat))

    links.extend(d_family.links)
    pars += 2

# State transition probabilities and initial state durations can be fixed
pi = np.array([1, 0])
T = np.array([[0, 1], [1,0]])

# Initialize Family
fam = HSMMFamily(
    pars,
    links,
    n_S,
    obs_fams=obs_families,
    d_fams=d_families,
    sid=np.array([0]),
    tid=None,
    T=T,
    pi=pi,
    D=D,
    M=M,
    starts_with_first=starts_with_first,
    ends_with_last=True,
    ends_in_last=False,
    n_cores=1,
)

# Fit model with N_u=30
bfgs_options = {"gtol": 0,
                "ftol": 1e-7,
                "maxcor": 30,
                "maxls": 100,
                "maxfun": 5000,
               }

energy_model = GSMM([*obs_formulas,*d_formulas],fam)
energy_model.fit(n_cores=n_cores*2,
                 method='qEFS',
                 bfgs_options = bfgs_options,
                 structured_qefs=False,
                 global_opt_qefs=True,
                 sample_hessian_options={"n_samples":30},
                 qEFS_final_memory_usage=None)


In [ ]:
# Visualize estimated smooth functions and latent state duration distributions
fig = plt.figure(figsize=(full_width,single_width),layout='constrained')
axs = fig.subplots(1,3,gridspec_kw=dict(wspace=0.1,hspace=0.1))
viterbi = fam.decode_viterbi(energy_model.coef,energy_model.coef_split_idx,
                             energy_model.get_ys(),energy_model.get_mmat())

axs[0].scatter(energy_dat["EurDol"],energy_dat["Price"],
               facecolor=[cmp(0.9) if s == 0 else cmp(0.1) for s in viterbi[0][1]],
               edgecolor=[cmp(0.99) if s == 0 else cmp(0.01) for s in viterbi[0][1]],
               alpha=0.2,s=3,linewidths=0.1)

pred_dat = pd.DataFrame({"EurDol":np.linspace(energy_dat["EurDol"].min(),
                                              energy_dat["EurDol"].max(),30)})

plot_fitted(pred_dat,["EurDol"],energy_model,ax=axs[0],response_scale=False,
            col=0.9,dist_par=0,ylim=[0,12])
plot_fitted(pred_dat,["EurDol"],energy_model,ax=axs[0],response_scale=False,
            col=0.1,dist_par=2,ylim=[0,12])

plot_fitted(pred_dat,["EurDol"],energy_model,ax=axs[1],response_scale=True,
            col=0.9,dist_par=1,ylim=[0.11,4])
plot_fitted(pred_dat,["EurDol"],energy_model,ax=axs[1],response_scale=True,
            col=0.1,dist_par=3,ylim=[0.11,4])

d_probs = fam.compute_od_probs(energy_model.coef,energy_model.coef_split_idx,
                               energy_model.get_ys(),energy_model.get_mmat(),
                               log=False)[0][1]
for si in range(2):
    
    pmf = d_probs[:D,si]
    support = np.linspace(1,D,pmf.shape[0])

    axs[2].fill([*support,*np.flip(support)],
                [*pmf,*np.zeros_like(pmf)],
                color=(cmp(0.9) if si == 0 else cmp(0.1)),
                alpha=0.5)
    axs[2].plot(support,pmf,color=(cmp(0.9) if si == 0 else cmp(0.1)),label=f"State {si+1}")

axs[2].legend()
axs[0].set_ylabel("$\\hat{\\mu}_j$",math_fontfamily=math_font,)
axs[1].set_ylabel("$\\hat{\\sigma}_j$",math_fontfamily=math_font,)
axs[2].set_ylabel("$p(d|S=j)$",math_fontfamily=math_font,)
axs[2].set_xlabel("State duration",fontweight='bold')

for axi,label in enumerate(["a)","b)","c)"]):
    axs[axi].text(-0.3,1.05,label,transform=axs[axi].transAxes,fontweight='bold',fontsize=plt.rcParams["axes.titlesize"])
plt.savefig(f"./results/plots/energy_prices.pdf",format="pdf", bbox_inches='tight')
plt.show()


## Tweedie Location, scale, and shape model of Mackerel Egg data

In [ ]:
# Load data
mackerel_dat = pd.read_csv("./data/mackerel.csv")

# Also coastline info for nicer plots
coast = pd.read_csv("./data/coast.csv")

# Create root and log variables
mackerel_dat["b.depth.r"] = np.sqrt(mackerel_dat["b.depth"].values)
mackerel_dat["log.vol"] = np.log(mackerel_dat["vol"].values)

# Handle nan's in predictors
nan_idx = np.any(np.isnan(mackerel_dat.loc[:,("T.20","Sal20")].values),axis=1)
mackerel_dat = mackerel_dat.loc[~nan_idx,:]

# Drop temperatures above 25 as done by Wood & Fasiolo
mackerel_dat = mackerel_dat.loc[mackerel_dat["T.20"].values < 25,:]

# Un-comment to exclude observation with extreme salinity level
# mackerel_dat = mackerel_dat.loc[mackerel_dat["Sal20"].values < 37,:] 

# Create model proposed by Wood & Fasiolo (2017) (almost):
#  - Replace Duchon spline with similarly complex tensor smooth
#  - Also cannot have offset for more general models in mssm currently,
#     so replace "log.vol" with a linear term l(["log.vol"])
formula_mu = Formula(lhs("count"),[i(),l(["log.vol"]),f(["lo","la"],te=True,nk=12),
                                   f(["T.20"]),f(["Sal20"]),f(["b.depth.r"]),
                                   ri("ship")],data=mackerel_dat)

formula_theta = Formula(lhs("count"),[i(),f(["b.depth.r"])],data=mackerel_dat)

formula_rho = Formula(lhs("count"),[i(),f(["b.depth.r"])],data=mackerel_dat)

formulas = [formula_mu,formula_theta,formula_rho]

# Fit model via EFS
mackerel_model = GAMMLSS(formulas,Tweedie())
mackerel_model.fit()

# Get residuals and partial prediction for partial residual plot
res = mackerel_model.get_resid()
sal20_pred = mackerel_model.predict([4],mackerel_dat)[0]

In [ ]:
# Visualize EFS results
fig = plt.figure(figsize=(full_width,2*single_width),layout='constrained')
axs = fig.subplots(2,3,gridspec_kw=dict(wspace=0.01,hspace=0.1)).flatten()

plot(mackerel_model,dist_par=0,which=[3],axs=[axs[0]],plot_exist=True,ylim=[-4,3])

axs[1].scatter(mackerel_dat["Sal20"].values,sal20_pred.reshape(-1,1) + res,color=cmp(0.7),alpha=0.5)
plot(mackerel_model,dist_par=0,which=[4],axs=[axs[1]],plot_exist=True,ylim=[-10,10])


plot(mackerel_model,plot_exist=True,lim_dist=0.05,ci=False,which=[2],axs=[axs[2]])
axs[2].plot(coast["lon"].values,coast["lat"].values,color="black",linewidth=1)

plot(mackerel_model,dist_par=0,which=[5],axs=[axs[3]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model,dist_par=1,axs=[axs[4]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model,dist_par=2,axs=[axs[5]],plot_exist=True,ylim=[-4,3])

for axi in [0,3,4,5]:
    axs[axi].set_yticks([-4,-3,-2,-1,0,1,2,3],
                        [-4,-3,-2,-1,0,1,2,3])
    
axs[0].set_ylabel("$f_{\\mu}(T20)$")
axs[1].set_ylabel("$f_{\\mu}(S20)$")
axs[0].set_xlabel("T20",fontweight="bold")
axs[1].set_xlabel("S20",fontweight="bold")
axs[3].set_ylabel("$f_{\\mu}\\left(\\sqrt{b.depth}\\right)$")
axs[4].set_ylabel("$f_{\\theta}\\left(\\sqrt{b.depth}\\right)$")
axs[5].set_ylabel("$f_{\\phi}\\left(\\sqrt{b.depth}\\right)$")

axs[3].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[4].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[5].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
plt.show()

In [ ]:
# Fit qEFS
mackerel_model2 = GSMM(formulas,GAMLSSGSMMFamily(3,mackerel_model.family))
mackerel_model2.fit(method='qEFS',structured_qefs=True,repara=True,
                    structured_qefs_budget=50,n_cores=1,
                    sample_hessian_options={"n_samples":50},
                    qEFS_final_memory_usage=None)

# Again get residuals and partial predictions
res2 = Tweedie().get_resid(mackerel_model2.get_ys()[0],*mackerel_model2.mus)
sal20_pred2 = mackerel_model2.predict([4],mackerel_dat)[0]

In [ ]:
# Visualize pqEFS results
fig = plt.figure(figsize=(full_width,2*single_width),layout='constrained')
axs = fig.subplots(2,3,gridspec_kw=dict(wspace=0.01,hspace=0.1)).flatten()

plot(mackerel_model2,dist_par=0,which=[3],axs=[axs[0]],plot_exist=True,ylim=[-4,3])

#axs[1].scatter(mackerel_dat["Sal20"].values,sal20_pred2.reshape(-1,1) + res2,color=cmp(0.7),alpha=0.5)
plot(mackerel_model,dist_par=0,which=[4],axs=[axs[1]],plot_exist=False,ylim=[-4,3],prov_cols=0.3)
plot(mackerel_model2,dist_par=0,which=[4],axs=[axs[1]],plot_exist=True,ylim=[-4,3])


plot(mackerel_model2,plot_exist=True,lim_dist=0.05,ci=False,which=[2],axs=[axs[2]])
axs[2].plot(coast["lon"].values,coast["lat"].values,color="black",linewidth=1)

plot(mackerel_model2,dist_par=0,which=[5],axs=[axs[3]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model2,dist_par=1,axs=[axs[4]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model2,dist_par=2,axs=[axs[5]],plot_exist=True,ylim=[-4,3])

for axi in [0,1,3,4,5]:
    axs[axi].set_yticks([-4,-3,-2,-1,0,1,2,3],
                        [-4,-3,-2,-1,0,1,2,3])

for axi,label in enumerate(["a","b","c","d","e","f"]):
    axs[axi].text(-0.3,1.05,f"{label})",transform=axs[axi].transAxes,fontweight="bold",fontsize=plt.rcParams["axes.titlesize"])
    
axs[0].set_ylabel("$f_{\\mu}(T20)$")
axs[1].set_ylabel("$f_{\\mu}(S20)$")
axs[0].set_xlabel("T20",fontweight="bold")
axs[1].set_xlabel("S20",fontweight="bold")
axs[3].set_ylabel("$f_{\\mu}\\left(\\sqrt{b.depth}\\right)$")
axs[4].set_ylabel("$f_{\\theta}\\left(\\sqrt{b.depth}\\right)$")
axs[5].set_ylabel("$f_{\\phi}\\left(\\sqrt{b.depth}\\right)$")

axs[3].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[4].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[5].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
plt.savefig("./results/plots/mackerel.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Fit qEFS with f(sal20) coef in set of exact hessian
sal20_idx = formulas[0].coef_idx_per_term[4]
fdcol = np.sort([*sal20_idx,*np_gen.choice([idx for idx in np.arange(mackerel_model2.coef.shape[0]).tolist() if idx not in sal20_idx],
                                            size=50-len(sal20_idx),
                                            replace=False)
                ]).tolist()

mackerel_model3 = GSMM(formulas,GAMLSSGSMMFamily(3,mackerel_model.family))
mackerel_model3.fit(method='qEFS',structured_qefs=True,repara=True,
                    structured_qefs_budget=fdcol,n_cores=1,
                    sample_hessian_options={"n_samples":50},
                    qEFS_final_memory_usage=None)

In [ ]:
# Visualize qEFS results one last time
fig = plt.figure(figsize=(full_width,2*single_width),layout='constrained')
axs = fig.subplots(2,3,gridspec_kw=dict(wspace=0.01,hspace=0.1)).flatten()

plot(mackerel_model3,dist_par=0,which=[3],axs=[axs[0]],plot_exist=True,ylim=[-4,3])

axs[1].scatter(mackerel_dat["Sal20"].values,sal20_pred.reshape(-1,1) + res,color=cmp(0.7),alpha=0.5)
plot(mackerel_model3,dist_par=0,which=[4],axs=[axs[1]],plot_exist=True,ylim=[-10,10])


plot(mackerel_model3,plot_exist=True,lim_dist=0.05,ci=False,which=[2],axs=[axs[2]])
axs[2].plot(coast["lon"].values,coast["lat"].values,color="black",linewidth=1)

plot(mackerel_model3,dist_par=0,which=[5],axs=[axs[3]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model3,dist_par=1,axs=[axs[4]],plot_exist=True,ylim=[-4,3])
plot(mackerel_model3,dist_par=2,axs=[axs[5]],plot_exist=True,ylim=[-4,3])

for axi in [0,3,4,5]:
    axs[axi].set_yticks([-4,-3,-2,-1,0,1,2,3],
                        [-4,-3,-2,-1,0,1,2,3])
    
axs[0].set_ylabel("$f_{\\mu}(T20)$")
axs[1].set_ylabel("$f_{\\mu}(S20)$")
axs[0].set_xlabel("T20",fontweight="bold")
axs[1].set_xlabel("S20",fontweight="bold")
axs[3].set_ylabel("$f_{\\mu}\\left(\\sqrt{b.depth}\\right)$")
axs[4].set_ylabel("$f_{\\theta}\\left(\\sqrt{b.depth}\\right)$")
axs[5].set_ylabel("$f_{\\phi}\\left(\\sqrt{b.depth}\\right)$")

axs[3].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[4].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
axs[5].set_xlabel("$\\sqrt{\\regular{b.depth}}$",fontweight="bold")
plt.show()